# DDInter → PharmPilot Interaction Bundle
Builds a `bundle-v1` SQLite from DDInter 2.0 and downloads it. Then install it in **Admin → Interaction Bundle**.

Run all cells top to bottom (Runtime → Run all). No edits required for a public repo; if the repo is private, authenticate the clone first (see cell 1).

In [ ]:
# 1. Dependencies + the PharmPilot package (for the SAME normalize() + schema constant).
!pip -q install pandas requests pyyaml
import os, sys
if not os.path.isdir('/content/pharmpilot'):
    !git clone --depth 1 https://github.com/sasha-dianat/PharmPilot.git /content/pharmpilot
sys.path.insert(0, '/content/pharmpilot')
# Private repo? Replace the clone above with a token URL, e.g.:
#   !git clone https://<USER>:<TOKEN>@github.com/sasha-dianat/PharmPilot.git /content/pharmpilot
from scripts.interaction_bundle.ddinter_builder import RawInteraction, build_rules, write_bundle
print('builder imported OK')


In [ ]:
# 2. Download DDInter 2.0 DDI CSVs (per ATC therapeutic category).
import pandas as pd, requests, io
BASE = 'https://ddinter2.scbdd.com/static/media/download/ddinter_downloads_code_'
CATEGORIES = ['A', 'B', 'D', 'H', 'L', 'P', 'R', 'V']
frames = []
for c in CATEGORIES:
    url = f'{BASE}{c}.csv'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    frames.append(pd.read_csv(io.StringIO(r.text)))
    print(f'  {c}: {len(frames[-1])} rows')
raw = pd.concat(frames, ignore_index=True)
print('total rows:', len(raw)); raw.head()


In [ ]:
# 3. Adapt DDInter columns (Drug_A, Drug_B, Level) -> RawInteraction rows.
def to_rows(df):
    for _, r in df.iterrows():
        yield RawInteraction(str(r.get('Drug_A', '')), str(r.get('Drug_B', '')),
                             str(r.get('Level', '')), str(r.get('Mechanism', '') or ''))
rows = list(to_rows(raw))
print('rows:', len(rows))


In [ ]:
# 4. Build + write the bundle (tokens via the engine's own normalize()).
rules = build_rules(rows)
meta = write_bundle(rules, 'bundle.sqlite', {'ddinter': '2.0'})
print(meta)


In [ ]:
# 5. Download the bundle, then install it in Admin -> Interaction Bundle.
from google.colab import files  # Colab only
files.download('bundle.sqlite')


**Next:** in PharmPilot, open *Dashboards → Interaction Bundle* and upload `bundle.sqlite`. It is validated + atomically installed + hot-reloaded; curated rules always win. DDInter `Level` (Major/Moderate/Minor) maps to engine severity; mechanisms default when absent.